In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Si, SEPD

This example demonstrates a Rietveld refinement of Si crystal
structure using time-of-flight neutron powder diffraction data from
SEPD at Argonne.

It also shows how to switch calculation engine and peak profile type.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='si')

### Set Space Group

In [4]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.coord_system_code = '2'

### Set Unit Cell

In [5]:
structure.cell.length_a = 5.431

### Set Atom Sites

In [6]:
structure.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0.125,
    fract_y=0.125,
    fract_z=0.125,
    adp_iso=0.5,
)

## 🔬 Define Experiment

This section shows how to add experiments, configure their
parameters, and link the structures defined in the previous step.

### Download Data

In [7]:
data_path = download_data('meas-si-sepd', destination='data')

Getting data...


Data 'meas-si-sepd': Si, SEPD (Argonne)


✅ Data 'meas-si-sepd' downloaded to '../../../data/meas-si-sepd.xye'


### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(
    name='sepd',
    data_path=data_path,
    beam_mode='time-of-flight',
)

### Set Instrument

In [9]:
expt.instrument.setup_twotheta_bank = 144.845
expt.instrument.calib_d_to_tof_offset = 0.0
expt.instrument.calib_d_to_tof_linear = 7476.91
expt.instrument.calib_d_to_tof_quadratic = -1.54

### Set Peak Profile

In [10]:
expt.peak.show_supported()

Peak types


,,Type,Description
1,,pseudo-voigt,TOF non-convoluted pseudo-Voigt profile
2,*,jorgensen,TOF Jorgensen profile: back-to-back exponentials ⊗ Gaussian
3,,jorgensen-von-dreele,TOF Jorgensen-Von Dreele profile: back-to-back exponentials ⊗ pseudo-Voigt
4,,double-jorgensen-von-dreele,TOF Double-Jorgensen-Von Dreele profile: double back-to-back exponentials ⊗ pseudo-Voigt (Z-Rietveld type0m)


In [11]:
expt.peak.type = 'jorgensen-von-dreele'

⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • broad_lorentz_gamma_0=0.0                                                                                                    
   • broad_lorentz_gamma_1=0.0                                                                                                    
   • broad_lorentz_gamma_2=0.0                                                                                                    
   • broad_lorentz_size_l=0.0                                                                                                     
   • broad_lorentz_strain_l=0.0                                                                                                   


Peak profile type for experiment 'sepd' changed to


jorgensen-von-dreele


In [12]:
expt.peak.broad_gauss_sigma_0 = 3.0148
expt.peak.broad_gauss_sigma_1 = 33.3451
expt.peak.broad_lorentz_gamma_1 = 2.5489
expt.peak.decay_beta_0 = 0.04221
expt.peak.decay_beta_1 = 0.00946
expt.peak.rise_alpha_1 = 0.5971

In [13]:
expt.peak.cutoff_fwhm = 10

### Set Background

In [14]:
expt.background.auto_estimate()

### Set Linked Structures

In [15]:
expt.linked_structures.create(structure_id='si', scale=600.0)

## 📦 Define Project

The project object is used to manage the structure, experiment, and
analysis.

### Create Project

In [16]:
project = Project(name='si_sepd')

### Add Structure

In [17]:
project.structures.add(structure)

### Add Experiment

In [18]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

### Display Structure

In [19]:
project.display.structure(struct_name='si')

Structure 🧩 'si' (Atom view type: 'covalent')


### Display Pattern

In [20]:
project.display.pattern(expt_name='sepd')
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 1/4

Set parameters to be refined.

In [21]:
structure.cell.length_a.free = True

expt.linked_structures['si'].scale.free = True
expt.instrument.calib_d_to_tof_offset.free = True

Show free parameters after selection.

In [22]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43100,,-inf,inf,Å
2,sepd,linked_structure,si,scale,600.00000,,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,0.00000,,-inf,inf,μs


#### Run Fitting

In [23]:
project.analysis.minimizer.type = 'bumps (lm)'

Current minimizer changed to


bumps (lm)


In [24]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),8.79
4,📏 Goodness-of-fit (reduced χ²),7.18
5,"📏 R-factor (Rf, %)",11.46
6,"📏 R-factor squared (Rf², %)",5.55
7,"📏 Weighted R-factor (wR, %)",4.09


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4310,5.4312,0.0001,0.00 % ↑
2,sepd,linked_structure,si,scale,,600.0000,612.1393,1.6329,2.02 % ↑
3,sepd,instrument,,d_to_tof_offset,μs,0.0000,-8.9744,0.0810,N/A


#### Display Pattern

In [25]:
project.display.pattern(expt_name='sepd')

In [26]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 2/4

Set more parameters to be refined.

In [27]:
for point in expt.background:
    point.intensity.free = True

Show free parameters after selection.

In [28]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43116,0.00006,-inf,inf,Å
2,sepd,linked_structure,si,scale,612.13932,1.63290,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,-8.97444,0.08101,-inf,inf,μs
4,sepd,background,1,intensity,213.55062,,-inf,inf,
5,sepd,background,2,intensity,117.61669,,-inf,inf,
6,sepd,background,3,intensity,147.70005,,-inf,inf,
7,sepd,background,4,intensity,122.26237,,-inf,inf,
8,sepd,background,5,intensity,163.04903,,-inf,inf,
9,sepd,background,6,intensity,124.58762,,-inf,inf,
10,sepd,background,7,intensity,120.30738,,-inf,inf,


#### Run Fitting

In [29]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),32.59
4,📏 Goodness-of-fit (reduced χ²),3.76
5,"📏 R-factor (Rf, %)",8.21
6,"📏 R-factor squared (Rf², %)",4.01
7,"📏 Weighted R-factor (wR, %)",2.70


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4312,5.4313,0.0000,0.00 % ↑
2,sepd,linked_structure,si,scale,,612.1393,631.4776,1.2200,3.16 % ↑
3,sepd,instrument,,d_to_tof_offset,μs,-8.9744,-9.1605,0.0568,2.07 % ↑
4,sepd,background,1,intensity,,213.5506,203.0531,0.4138,4.92 % ↓
5,sepd,background,2,intensity,,117.6167,102.9386,0.4497,12.48 % ↓
6,sepd,background,3,intensity,,147.7001,125.1584,0.8784,15.26 % ↓
7,sepd,background,4,intensity,,122.2624,119.5886,0.9714,2.19 % ↓
8,sepd,background,5,intensity,,163.0490,129.5253,2.7603,20.56 % ↓
9,sepd,background,6,intensity,,124.5876,122.8643,1.6916,1.38 % ↓
10,sepd,background,7,intensity,,120.3074,121.2266,1.7762,0.76 % ↑


#### Display Pattern

In [30]:
project.display.pattern(expt_name='sepd')

In [31]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 3/4

Fix background points.

In [32]:
for point in expt.background:
    point.intensity.free = False

Set more parameters to be refined.

In [33]:
expt.peak.broad_gauss_sigma_0.free = True
expt.peak.broad_gauss_sigma_1.free = True
expt.peak.broad_lorentz_gamma_1.free = True

Show free parameters after selection.

In [34]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43131,0.00004,-inf,inf,Å
2,sepd,linked_structure,si,scale,631.47760,1.21996,-inf,inf,
3,sepd,peak,,broad_lorentz_gamma_1,2.54890,,-inf,inf,μs/Å
4,sepd,peak,,broad_gauss_sigma_0,3.01480,,-inf,inf,μs²
5,sepd,peak,,broad_gauss_sigma_1,33.34510,,-inf,inf,μs/Å
6,sepd,instrument,,d_to_tof_offset,-9.16049,0.05683,-inf,inf,μs


#### Run Fitting

In [35]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),18.54
4,📏 Goodness-of-fit (reduced χ²),3.73
5,"📏 R-factor (Rf, %)",8.27
6,"📏 R-factor squared (Rf², %)",4.11
7,"📏 Weighted R-factor (wR, %)",2.80


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4313,5.4313,0.0000,0.00 % ↑
2,sepd,linked_structure,si,scale,,631.4776,629.4282,1.2713,0.32 % ↓
3,sepd,peak,,broad_lorentz_gamma_1,μs/Å,2.5489,2.1873,0.0739,14.19 % ↓
4,sepd,peak,,broad_gauss_sigma_0,μs²,3.0148,4.0830,0.3556,35.43 % ↑
5,sepd,peak,,broad_gauss_sigma_1,μs/Å,33.3451,34.2206,0.6823,2.63 % ↑
6,sepd,instrument,,d_to_tof_offset,μs,-9.1605,-9.1720,0.0579,0.13 % ↑


#### Display Pattern

In [36]:
project.display.pattern(expt_name='sepd')

In [37]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 4/4

Set more parameters to be refined.

In [38]:
structure.atom_sites['Si'].adp_iso.free = True

expt.peak.decay_beta_0.free = True
expt.peak.decay_beta_1.free = True
expt.peak.rise_alpha_1.free = True

Show free parameters after selection.

In [39]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43132,0.00004,-inf,inf,Å
2,si,atom_site,Si,adp_iso,0.50000,,-inf,inf,Å²
3,sepd,linked_structure,si,scale,629.42817,1.27125,-inf,inf,
4,sepd,peak,,rise_alpha_1,0.59710,,-inf,inf,μs/Å
5,sepd,peak,,decay_beta_0,0.04221,,-inf,inf,μs
6,sepd,peak,,decay_beta_1,0.00946,,-inf,inf,μs/Å
7,sepd,peak,,broad_lorentz_gamma_1,2.18730,0.07391,-inf,inf,μs/Å
8,sepd,peak,,broad_gauss_sigma_0,4.08299,0.35562,-inf,inf,μs²
9,sepd,peak,,broad_gauss_sigma_1,34.22059,0.68229,-inf,inf,μs/Å
10,sepd,instrument,,d_to_tof_offset,-9.17202,0.05786,-inf,inf,μs


#### Run Fitting

In [40]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),53.13
4,📏 Goodness-of-fit (reduced χ²),3.62
5,"📏 R-factor (Rf, %)",8.21
6,"📏 R-factor squared (Rf², %)",4.10
7,"📏 Weighted R-factor (wR, %)",2.90


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4313,5.4305,0.0007,0.01 % ↓
2,si,atom_site,Si,adp_iso,Å²,0.5000,0.5167,0.0036,3.34 % ↑
3,sepd,linked_structure,si,scale,,629.4282,633.6833,1.7658,0.68 % ↑
4,sepd,peak,,rise_alpha_1,μs/Å,0.5971,0.8662,0.7628,45.06 % ↑
5,sepd,peak,,decay_beta_0,μs,0.0422,0.0408,0.0003,3.32 % ↓
6,sepd,peak,,decay_beta_1,μs/Å,0.0095,0.0123,0.0002,29.86 % ↑
7,sepd,peak,,broad_lorentz_gamma_1,μs/Å,2.1873,2.3010,0.1007,5.20 % ↑
8,sepd,peak,,broad_gauss_sigma_0,μs²,4.0830,5.7797,0.4361,41.55 % ↑
9,sepd,peak,,broad_gauss_sigma_1,μs/Å,34.2206,34.2201,2.3831,0.00 % ↓
10,sepd,instrument,,d_to_tof_offset,μs,-9.1720,-8.4319,0.0814,8.07 % ↓


#### Display Correlations

In [41]:
project.display.fit.correlations()

#### Display Pattern

In [42]:
project.display.pattern(expt_name='sepd')

In [43]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

In [44]:
project.display.pattern(expt_name='sepd', x='d_spacing')

## 💾 Save Project

In [45]:
project.save_as(dir_path='projects/refine-si-sepd')

Saving project 📦 'si_sepd' to '../../../projects/refine-si-sepd'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 si.edi


├── 📁 experiments/


│   └── 📄 sepd.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 si_sepd.html
